In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# products

In [3]:
file_path = '/content/drive/MyDrive/vinuni_datathon2026/cleaned_datasets/products_cleaned.csv'
df_returns = pd.read_csv(file_path)

# Hiển thị 5 dòng đầu tiên
display(df_returns.head())

,product_id,product_name,category,segment,size,color,price,cogs,gross_profit,margin
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875,1354.807125,0.1225
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254,4129.205759,0.4336
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278,4579.713880,0.2871
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954,7180.544345,0.4558
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406,1702.764130,0.1080


# customers

In [4]:
file_path = '/content/drive/MyDrive/vinuni_datathon2026/cleaned_datasets/customers_cleaned.csv'
df_customers = pd.read_csv(file_path)

# Hiển thị 5 dòng đầu tiên
display(df_customers.head())

,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search


In [5]:
df_customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   customer_id          121930 non-null  int64 
 1   zip                  121930 non-null  int64 
 2   city                 121930 non-null  object
 3   signup_date          121930 non-null  object
 4   gender               121930 non-null  object
 5   age_group            121930 non-null  object
 6   acquisition_channel  121930 non-null  object
dtypes: int64(2), object(5)
memory usage: 6.5+ MB


In [6]:
# Bỏ cột zip
df_customers.drop(columns=['zip'], inplace=True)

In [7]:
df_customers['signup_date'] = pd.to_datetime(df_customers['signup_date'])
before_df_customers = df_customers[df_customers['signup_date'] < '2012-07-04']
display(before_df_customers.head())

,customer_id,city,signup_date,gender,age_group,acquisition_channel
97,125,Phu Ly,2012-04-24,Male,35-44,social_media
293,395,Hai Phong,2012-04-06,Female,25-34,social_media
1977,2544,Viet Tri,2012-05-03,Female,35-44,paid_search
1994,2564,Bac Giang,2012-06-04,Male,45-54,direct
2149,2756,Ha Long,2012-06-28,Male,25-34,paid_search


## Tạo DataFrame tổng hợp khách hàng theo thời gian (từ 04/07/2012 đến 31/12/2022)

In [8]:
import pandas as pd
import numpy as np

# Đảm bảo cột 'signup_date' là kiểu datetime
df_customers['signup_date'] = pd.to_datetime(df_customers['signup_date'])

# Định nghĩa khoảng ngày cho DataFrame cuối cùng
start_df_date = pd.to_datetime('2012-07-04')
end_df_date = pd.to_datetime('2022-12-31')

# Tạo một DataFrame với tất cả các ngày trong khoảng thời gian mong muốn
full_output_date_range = pd.DataFrame({'date': pd.date_range(start=start_df_date, end=end_df_date)})

# --- 1. Xử lý khách hàng trước ngày start_df_date (trạng thái ban đầu) ---
df_initial_customers = df_customers[df_customers['signup_date'] < start_df_date].copy()

# One-hot encode toàn bộ df_customers để đảm bảo tất cả các cột category có thể có được tạo
cols_to_onehot = ['gender', 'age_group', 'acquisition_channel']
df_customers_all_ohe = pd.get_dummies(df_customers, columns=cols_to_onehot, prefix=cols_to_onehot)

# Lấy tổng ban đầu của các cột đã one-hot encoded từ khách hàng trước ngày start_df_date
initial_ohe_frame = df_customers_all_ohe[df_customers_all_ohe['signup_date'] < start_df_date]
initial_encoded_sums = initial_ohe_frame.drop(columns=['customer_id', 'city', 'signup_date'], errors='ignore').sum(numeric_only=True)
initial_encoded_sums_dict = initial_encoded_sums.to_dict()

# Số lượng khách hàng lũy kế ban đầu
initial_customer_cumulative = df_initial_customers['customer_id'].nunique()

# Số lượng thành phố duy nhất lũy kế ban đầu
initial_unique_cities_cumulative_set = set(df_initial_customers['city'].unique())

# --- 2. Chuẩn bị dữ liệu cho tổng hợp hàng ngày trong khoảng thời gian mục tiêu ---
df_customers_in_period_ohe = df_customers_all_ohe[(df_customers_all_ohe['signup_date'] >= start_df_date) & (df_customers_all_ohe['signup_date'] <= end_df_date)].copy()

# Lấy tất cả tên cột đã one-hot encoded
ohe_cols = [col for col in df_customers_all_ohe.columns if any(prefix in col for prefix in cols_to_onehot)]

# --- 3. Tổng hợp hàng ngày ---
# Đếm số lượng đăng ký hàng ngày
daily_signups_count = df_customers_in_period_ohe.groupby('signup_date').size().reset_index(name='customer_daily_signup')
daily_signups_count = daily_signups_count.rename(columns={'signup_date': 'date'})

# Tổng hàng ngày của các tính năng đã one-hot encoded
daily_ohe_sums = df_customers_in_period_ohe.groupby('signup_date')[ohe_cols].sum().reset_index()
daily_ohe_sums = daily_ohe_sums.rename(columns={'signup_date': 'date'})

# Gộp các tổng và đếm hàng ngày
df_daily_agg = pd.merge(daily_signups_count, daily_ohe_sums, on='date', how='outer')

# Gộp với full_output_date_range để đảm bảo tất cả các ngày đều có, điền NaN bằng 0
df_daily_agg = pd.merge(full_output_date_range, df_daily_agg, on='date', how='left').fillna(0)
df_daily_agg = df_daily_agg.sort_values('date').reset_index(drop=True)

# --- 4. Tính toán các giá trị lũy kế ---
df_final_aggregated = df_daily_agg.copy()

# Số lượng khách hàng lũy kế (bao gồm khách hàng trước start_df_date và các đăng ký đến ngày trước đó)
df_final_aggregated['customer_cumulative'] = df_final_aggregated['customer_daily_signup'].shift(1).fillna(0).cumsum() + initial_customer_cumulative

# Số lượng lũy kế cho các cột đã one-hot encoded (đến ngày trước đó)
for col in ohe_cols:
    df_final_aggregated[col + '_cumulative'] = df_final_aggregated[col].shift(1).fillna(0).cumsum() + initial_encoded_sums_dict.get(col, 0)
    df_final_aggregated = df_final_aggregated.drop(columns=[col]) # Xóa cột tổng hàng ngày, giữ cột lũy kế

# --- 5. Số lượng thành phố duy nhất lũy kế ---
# Chuẩn bị dữ liệu thành phố để tính toán lũy kế
all_cities_data = df_customers[['signup_date', 'city']].sort_values('signup_date').copy()

cumulative_unique_cities_list = []
# Khởi tạo tập hợp các thành phố đã thấy với các thành phố từ trước start_df_date
seen_cities_set = initial_unique_cities_cumulative_set.copy()

# Tìm chỉ mục bắt đầu của khách hàng có signup_date >= start_df_date
current_customer_idx = all_cities_data[all_cities_data['signup_date'] >= start_df_date].index.min()
if pd.isna(current_customer_idx):
    current_customer_idx = len(all_cities_data)
else:
    current_customer_idx = int(current_customer_idx)

# Lặp qua từng ngày trong DataFrame kết quả cuối cùng
for date in df_final_aggregated['date']:
    # Thêm các thành phố từ khách hàng đăng ký trước ngày hiện tại (không bao gồm ngày hiện tại)
    while current_customer_idx < len(all_cities_data) and \
          all_cities_data.iloc[current_customer_idx]['signup_date'] < date:
        seen_cities_set.add(all_cities_data.iloc[current_customer_idx]['city'])
        current_customer_idx += 1
    cumulative_unique_cities_list.append(len(seen_cities_set))

df_final_aggregated['unique_cities_cumulative'] = cumulative_unique_cities_list

# Sắp xếp lại các cột để dễ đọc hơn
final_columns_order = ['date', 'customer_cumulative', 'customer_daily_signup', 'unique_cities_cumulative'] + \
                      sorted([col for col in df_final_aggregated.columns if 'gender_' in col and 'cumulative' in col]) + \
                      sorted([col for col in df_final_aggregated.columns if 'age_group_' in col and 'cumulative' in col]) + \
                      sorted([col for col in df_final_aggregated.columns if 'acquisition_channel_' in col and 'cumulative' in col])

df_final_aggregated = df_final_aggregated[final_columns_order]

display(df_final_aggregated.head())
display(df_final_aggregated.tail())

,date,customer_cumulative,customer_daily_signup,unique_cities_cumulative,gender_Female_cumulative,gender_Male_cumulative,gender_Non-binary_cumulative,age_group_18-24_cumulative,age_group_25-34_cumulative,age_group_35-44_cumulative,age_group_45-54_cumulative,age_group_55+_cumulative,acquisition_channel_direct_cumulative,acquisition_channel_email_campaign_cumulative,acquisition_channel_organic_search_cumulative,acquisition_channel_paid_search_cumulative,acquisition_channel_referral_cumulative,acquisition_channel_social_media_cumulative
0,2012-07-04,238.0,3.0,41,111.0,119.0,8.0,35.0,62.0,74.0,44.0,23.0,18.0,24.0,72.0,47.0,16.0,61.0
1,2012-07-05,241.0,5.0,41,112.0,121.0,8.0,36.0,62.0,76.0,44.0,23.0,18.0,24.0,72.0,48.0,17.0,62.0
2,2012-07-06,246.0,0.0,41,115.0,123.0,8.0,36.0,64.0,77.0,45.0,24.0,18.0,24.0,74.0,49.0,17.0,64.0
3,2012-07-07,246.0,4.0,41,115.0,123.0,8.0,36.0,64.0,77.0,45.0,24.0,18.0,24.0,74.0,49.0,17.0,64.0
4,2012-07-08,250.0,4.0,41,119.0,123.0,8.0,36.0,65.0,78.0,47.0,24.0,19.0,24.0,76.0,50.0,17.0,64.0


,date,customer_cumulative,customer_daily_signup,unique_cities_cumulative,gender_Female_cumulative,gender_Male_cumulative,gender_Non-binary_cumulative,age_group_18-24_cumulative,age_group_25-34_cumulative,age_group_35-44_cumulative,age_group_45-54_cumulative,age_group_55+_cumulative,acquisition_channel_direct_cumulative,acquisition_channel_email_campaign_cumulative,acquisition_channel_organic_search_cumulative,acquisition_channel_paid_search_cumulative,acquisition_channel_referral_cumulative,acquisition_channel_social_media_cumulative
3828,2022-12-27,121627.0,60.0,42,59502.0,57308.0,4817.0,17008.0,36248.0,31837.0,23124.0,13410.0,9780.0,14626.0,36358.0,24228.0,12238.0,24397.0
3829,2022-12-28,121687.0,65.0,42,59534.0,57333.0,4820.0,17015.0,36264.0,31851.0,23135.0,13422.0,9792.0,14637.0,36375.0,24238.0,12241.0,24404.0
3830,2022-12-29,121752.0,66.0,42,59566.0,57363.0,4823.0,17026.0,36288.0,31865.0,23143.0,13430.0,9796.0,14651.0,36394.0,24248.0,12248.0,24415.0
3831,2022-12-30,121818.0,51.0,42,59594.0,57396.0,4828.0,17030.0,36312.0,31885.0,23153.0,13438.0,9797.0,14661.0,36416.0,24262.0,12256.0,24426.0
3832,2022-12-31,121869.0,61.0,42,59616.0,57424.0,4829.0,17035.0,36326.0,31900.0,23159.0,13449.0,9801.0,14666.0,36431.0,24273.0,12264.0,24434.0


# geography

# promotions

In [9]:
file_path = '/content/drive/MyDrive/vinuni_datathon2026/cleaned_datasets/promotions_cleaned.csv'
df_promotions= pd.read_csv(file_path)

# Hiển thị 5 dòng đầu tiên
display(df_promotions.head())

,Unnamed: 0,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,0,PROMO-0001,Spring Sale 2013,percentage,12.0,2013-03-18,2013-04-17,all,email,1,0
1,1,PROMO-0002,Mid-Year Sale 2013,percentage,18.0,2013-06-23,2013-07-22,all,online,0,0
2,2,PROMO-0003,Fall Launch 2013,percentage,10.0,2013-08-30,2013-10-02,all,email,0,0
3,3,PROMO-0004,Year-End Sale 2013,percentage,20.0,2013-11-18,2014-01-02,all,all_channels,0,50000
4,4,PROMO-0005,Urban Blowout 2013,fixed,50.0,2013-07-30,2013-09-02,Streetwear,online,0,150000


In [10]:
# Drop cột Unnamed
# df_promotions.drop(columns=['Unnamed: 0'], inplace=True)
# Cột 'Unnamed: 0' không tồn tại trong df_promotions, dòng này đã được comment lại.

In [11]:
df_promotions

,Unnamed: 0,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,0,PROMO-0001,Spring Sale 2013,percentage,12.0,2013-03-18,2013-04-17,all,email,1,0
1,1,PROMO-0002,Mid-Year Sale 2013,percentage,18.0,2013-06-23,2013-07-22,all,online,0,0
2,2,PROMO-0003,Fall Launch 2013,percentage,10.0,2013-08-30,2013-10-02,all,email,0,0
3,3,PROMO-0004,Year-End Sale 2013,percentage,20.0,2013-11-18,2014-01-02,all,all_channels,0,50000
4,4,PROMO-0005,Urban Blowout 2013,fixed,50.0,2013-07-30,2013-09-02,Streetwear,online,0,150000
5,5,PROMO-0006,Rural Special 2013,percentage,15.0,2013-01-31,2013-03-01,Outdoor,in_store,0,0
6,6,PROMO-0007,Spring Sale 2014,percentage,12.0,2014-03-18,2014-04-17,all,email,1,0
7,7,PROMO-0008,Mid-Year Sale 2014,percentage,18.0,2014-06-23,2014-07-22,all,social_media,0,0
8,8,PROMO-0009,Fall Launch 2014,percentage,10.0,2014-08-30,2014-10-01,all,all_channels,0,100000
9,9,PROMO-0010,Year-End Sale 2014,percentage,20.0,2014-11-19,2015-01-02,all,all_channels,0,100000


In [12]:
import pandas as pd
import numpy as np

# 1. Prepare df_promotions
df_promotions_processed = df_promotions.copy()
df_promotions_processed['start_date'] = pd.to_datetime(df_promotions_processed['start_date'])
df_promotions_processed['end_date'] = pd.to_datetime(df_promotions_processed['end_date'])

# Create flag_min_order_value (1 if min_order_value exists and is > 0, else 0)
df_promotions_processed['flag_min_order_value'] = df_promotions_processed['min_order_value'].apply(lambda x: 1 if pd.notna(x) and x > 0 else 0)

# Drop 'stackable_flag' if it exists (it doesn't in the current df_promotions, but good practice)
if 'stackable_flag' in df_promotions_processed.columns:
    df_promotions_processed = df_promotions_processed.drop(columns=['stackable_flag'])

# One-hot encode categorical columns
cols_to_onehot_promo = ['promo_type', 'status', 'target_audience']
# Filter out columns that do not exist in the DataFrame
cols_to_onehot_promo = [col for col in cols_to_onehot_promo if col in df_promotions_processed.columns]

df_promotions_ohe = pd.get_dummies(df_promotions_processed, columns=cols_to_onehot_promo, prefix=cols_to_onehot_promo)

In [13]:
# 2. Generate a daily DataFrame of active promotions
# Create a list of all active (date, promo_id) pairs
all_active_promo_dates = []
for index, row in df_promotions_ohe.iterrows():
    promo_id = row['promo_id']
    start_dt = row['start_date']
    end_dt = row['end_date']
    for date in pd.date_range(start=start_dt, end=end_dt):
        all_active_promo_dates.append({'date': date, 'promo_id': promo_id})

df_active_promo_dates = pd.DataFrame(all_active_promo_dates)

# Define the full date range for the final output DataFrame
start_df_date = pd.to_datetime('2012-07-04')
end_df_date = pd.to_datetime('2022-12-31')
full_date_range_df = pd.DataFrame({'date': pd.date_range(start=start_df_date, end=end_df_date)})

# Filter active promo dates to be within the desired range
df_active_promo_dates_filtered = df_active_promo_dates[
    (df_active_promo_dates['date'] >= start_df_date) &
    (df_active_promo_dates['date'] <= end_df_date)
]

In [14]:
# Merge with df_promotions_ohe to get all promo details for each active day
# Select only relevant columns from df_promotions_ohe to avoid duplicates/unnecessary columns
promo_details_cols = ['promo_id', 'discount_value', 'flag_min_order_value'] + \
                     [col for col in df_promotions_ohe.columns if any(prefix in col for prefix in cols_to_onehot_promo)]

df_daily_promotions_detailed = pd.merge(df_active_promo_dates_filtered,
                                        df_promotions_ohe[promo_details_cols],
                                        on='promo_id',
                                        how='left')

# 3. Aggregate daily promotion data
aggregation_dict = {
    'promo_id': lambda x: 1 if x.nunique() > 0 else 0, # promo_flag: 1 if any promo is active
    'discount_value': 'mean', # avg_discount_value
    'flag_min_order_value': 'max' # 1 if any active promo has min_order_value
}

# Add one-hot encoded columns to aggregation dict with 'max' to indicate presence
ohe_promo_agg_cols = [col for col in df_promotions_ohe.columns if any(prefix in col for prefix in cols_to_onehot_promo)]
for ohe_col in ohe_promo_agg_cols:
    aggregation_dict[ohe_col] = 'max'

df_promotions_agg_daily = df_daily_promotions_detailed.groupby('date').agg(aggregation_dict).reset_index()

# Rename promo_id aggregate to promo_flag and discount_value to avg_discount_value
df_promotions_agg_daily = df_promotions_agg_daily.rename(columns={'promo_id': 'promo_flag', 'discount_value': 'avg_discount_value'})

In [15]:
# 4. Create final df_promo_features DataFrame
df_promo_features = pd.merge(full_date_range_df, df_promotions_agg_daily, on='date', how='left').fillna(0)

# Convert appropriate columns to integer type
df_promo_features['promo_flag'] = df_promo_features['promo_flag'].astype(int)
df_promo_features['flag_min_order_value'] = df_promo_features['flag_min_order_value'].astype(int)
for col in ohe_promo_agg_cols:
    df_promo_features[col] = df_promo_features[col].astype(int)

# Display the head and tail of the resulting DataFrame
display(df_promo_features.head())
display(df_promo_features.tail())

,date,promo_flag,avg_discount_value,flag_min_order_value,promo_type_fixed,promo_type_percentage
0,2012-07-04,0,0.0,0,0,0
1,2012-07-05,0,0.0,0,0,0
2,2012-07-06,0,0.0,0,0,0
3,2012-07-07,0,0.0,0,0,0
4,2012-07-08,0,0.0,0,0,0


,date,promo_flag,avg_discount_value,flag_min_order_value,promo_type_fixed,promo_type_percentage
3828,2022-12-27,1,20.0,1,0,1
3829,2022-12-28,1,20.0,1,0,1
3830,2022-12-29,1,20.0,1,0,1
3831,2022-12-30,1,20.0,1,0,1
3832,2022-12-31,1,20.0,1,0,1


## Hợp nhất dữ liệu MASTER

In [16]:
daily_master_df = pd.merge(df_final_aggregated, df_promo_features, on='date', how='left')

print("5 hàng đầu của daily_master_df:")
display(daily_master_df.head())

print("\nThông tin về daily_master_df:")
daily_master_df.info()

5 hàng đầu của daily_master_df:


,date,customer_cumulative,customer_daily_signup,unique_cities_cumulative,gender_Female_cumulative,gender_Male_cumulative,gender_Non-binary_cumulative,age_group_18-24_cumulative,age_group_25-34_cumulative,age_group_35-44_cumulative,...,acquisition_channel_email_campaign_cumulative,acquisition_channel_organic_search_cumulative,acquisition_channel_paid_search_cumulative,acquisition_channel_referral_cumulative,acquisition_channel_social_media_cumulative,promo_flag,avg_discount_value,flag_min_order_value,promo_type_fixed,promo_type_percentage
0,2012-07-04,238.0,3.0,41,111.0,119.0,8.0,35.0,62.0,74.0,...,24.0,72.0,47.0,16.0,61.0,0,0.0,0,0,0
1,2012-07-05,241.0,5.0,41,112.0,121.0,8.0,36.0,62.0,76.0,...,24.0,72.0,48.0,17.0,62.0,0,0.0,0,0,0
2,2012-07-06,246.0,0.0,41,115.0,123.0,8.0,36.0,64.0,77.0,...,24.0,74.0,49.0,17.0,64.0,0,0.0,0,0,0
3,2012-07-07,246.0,4.0,41,115.0,123.0,8.0,36.0,64.0,77.0,...,24.0,74.0,49.0,17.0,64.0,0,0.0,0,0,0
4,2012-07-08,250.0,4.0,41,119.0,123.0,8.0,36.0,65.0,78.0,...,24.0,76.0,50.0,17.0,64.0,0,0.0,0,0,0



Thông tin về daily_master_df:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 23 columns):
 #   Column                                         Non-Null Count  Dtype         
---  ------                                         --------------  -----         
 0   date                                           3833 non-null   datetime64[ns]
 1   customer_cumulative                            3833 non-null   float64       
 2   customer_daily_signup                          3833 non-null   float64       
 3   unique_cities_cumulative                       3833 non-null   int64         
 4   gender_Female_cumulative                       3833 non-null   float64       
 5   gender_Male_cumulative                         3833 non-null   float64       
 6   gender_Non-binary_cumulative                   3833 non-null   float64       
 7   age_group_18-24_cumulative                     3833 non-null   float64       
 8   age_group_25-34_cumulative 